# Notebook 2 — Graph Construction
Builds one contact graph per population file.
Run on **Colab CPU**. Takes ~20 minutes per graph, ~3.5 hours total.
Output: `graph_seed{N}.pt` for each population seed.

In [ ]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/epidemic_project'
print('Files in data/:')
for f in sorted(os.listdir(f'{BASE}/data')):
    print(' ', f)

In [ ]:
# Cell 2 — Install
!pip install -q torch-geometric pyarrow pandas numpy scikit-learn torch

In [ ]:
# Cell 3 — Imports
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from collections import defaultdict
from itertools import combinations
import math
print('Imports OK')

In [ ]:
# Cell 4 — Graph builder function
def build_graph(df, threshold_pct=0.30):
    """
    Build a multi-layer contact graph from a population dataframe.
    threshold_pct: top fraction of edges per node to keep (0.30 = top 30%)
    Returns a PyG Data object.
    """
    N = len(df)
    print(f'  Building graph for {N} nodes...')

    # --- Normalise features ---
    feature_cols = ['age','sex','ses','social_freq','mobility','comorbidity','vaccinated']
    scaler = MinMaxScaler()
    norm   = scaler.fit_transform(df[feature_cols].values)
    norm_df = pd.DataFrame(norm, columns=feature_cols)

    # Zone sin/cos encoding
    zone_sin = np.sin(2 * np.pi * df['zone'].values / 20)
    zone_cos = np.cos(2 * np.pi * df['zone'].values / 20)
    X = np.column_stack([norm_df.values, zone_sin, zone_cos])  # (N, 9)

    edge_weights = defaultdict(float)

    # --- Layer 1: Household edges (weight 1.0) ---
    print('  Layer 1: household edges...')
    for hid in df['household_id'].unique():
        members = df[df['household_id'] == hid]['node_id'].values
        for u, v in combinations(members, 2):
            edge_weights[(int(u), int(v))] += 1.0
            edge_weights[(int(v), int(u))] += 1.0

    # --- Layer 2: Workplace / school edges ---
    print('  Layer 2: workplace/school edges...')
    for col in ['workplace_id', 'school_id']:
        for gid in df[col].unique():
            if gid == -1: continue
            members = df[df[col] == gid]['node_id'].values
            sz = len(members)
            w  = 0.4 / math.log(sz + 2)
            # Cap at 60 to avoid O(n^2) explosion in large workplaces
            cap = members[:60]
            for u, v in combinations(cap, 2):
                edge_weights[(int(u), int(v))] += w
                edge_weights[(int(v), int(u))] += w

    # --- Layer 3: Geographic KNN similarity ---
    print('  Layer 3: geographic similarity edges...')
    feat_sim = norm_df[['age','ses','mobility']].values
    for zone in range(1, 21):
        idx = df[df['zone'] == zone].index.values
        if len(idx) < 2: continue
        feats = feat_sim[idx]
        k = min(12, len(idx) - 1)
        nbrs = NearestNeighbors(n_neighbors=k+1, metric='euclidean').fit(feats)
        dists, indices = nbrs.kneighbors(feats)
        max_d = dists[:, 1:].max() + 1e-8
        for li, (ds, idxs) in enumerate(zip(dists[:,1:], indices[:,1:])):
            u = int(idx[li])
            for d, lj in zip(ds, idxs):
                v = int(idx[lj])
                w = 0.3 * (1 - d / max_d)
                edge_weights[(u, v)] += w
                edge_weights[(v, u)] += w

    # --- Degree-preserving threshold ---
    print(f'  Thresholding (top {int(threshold_pct*100)}% per node)...')
    node_edges = defaultdict(list)
    for (u, v), w in edge_weights.items():
        node_edges[u].append((v, w))

    final_src, final_dst, final_w = [], [], []
    for u, nbrs in node_edges.items():
        nbrs_sorted = sorted(nbrs, key=lambda x: -x[1])
        keep = max(3, int(len(nbrs_sorted) * threshold_pct))
        for v, w in nbrs_sorted[:keep]:
            final_src.append(u)
            final_dst.append(v)
            final_w.append(w)

    edge_index = torch.tensor([final_src, final_dst], dtype=torch.long)
    edge_attr  = torch.tensor(final_w, dtype=torch.float).unsqueeze(1)
    node_x     = torch.tensor(X, dtype=torch.float)
    node_pos   = torch.tensor(df[['lat','lng']].values, dtype=torch.float)
    node_zone  = torch.tensor(df['zone'].values, dtype=torch.long)
    pop_seed   = int(df['pop_seed'].iloc[0])

    data = Data(
        x          = node_x,
        edge_index = edge_index,
        edge_attr  = edge_attr,
        pos        = node_pos,
        zone       = node_zone,
        pop_seed   = pop_seed,
    )
    data.num_nodes = N

    # Degree stats
    src_t  = edge_index[0]
    degree = torch.zeros(N, dtype=torch.long)
    degree.scatter_add_(0, src_t, torch.ones(src_t.shape[0], dtype=torch.long))
    print(f'  Nodes={N}, Edges={edge_index.shape[1]}, '
          f'MeanDeg={degree.float().mean():.1f}, MinDeg={degree.min().item()}')
    return data

In [ ]:
# Cell 5 — Build all 10 graphs
# Varying threshold slightly across seeds adds more diversity
SEEDS = [0, 7, 13, 21, 42, 55, 77, 88, 99, 111]
THRESHOLDS = [0.30, 0.28, 0.32, 0.30, 0.30, 0.28, 0.32, 0.30, 0.28, 0.30]

for seed, thresh in zip(SEEDS, THRESHOLDS):
    pop_file = f'{BASE}/data/population_seed{seed}.parquet'
    out_file = f'{BASE}/graphs/graph_seed{seed}.pt'

    if os.path.exists(out_file):
        print(f'Seed {seed}: already exists, skipping.')
        continue

    print(f'\nSeed {seed} (threshold={thresh}):')
    df   = pd.read_parquet(pop_file)
    data = build_graph(df, threshold_pct=thresh)
    torch.save(data, out_file)
    print(f'  Saved to {out_file}')

print('\nAll graphs built.')

In [ ]:
# Cell 6 — Verify all graphs
for seed in SEEDS:
    g = torch.load(f'{BASE}/graphs/graph_seed{seed}.pt')
    src = g.edge_index[0]
    deg = torch.zeros(g.num_nodes, dtype=torch.long)
    deg.scatter_add_(0, src, torch.ones(src.shape[0], dtype=torch.long))
    print(f'Seed {seed:3d}: nodes={g.num_nodes}, edges={g.edge_index.shape[1]:6d}, '
          f'mean_deg={deg.float().mean():.1f}, x_shape={g.x.shape}')